In [42]:
import os.path

import torch
import transformers

from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [43]:
device = "cuda" if torch.cuda.is_available() else "cpu"
models_dir = os.path.expanduser('/home/jupyter/datasphere/project')

In [54]:
model_name = os.path.join(models_dir, 'finetune_rugpt_text_to_prompt_batch_4_aug_data')
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({'bos_token': '<s>', 'eos_token': '</s>', 'pad_token': '<pad>'})
model = GPT2LMHeadModel.from_pretrained(model_name)

model.to(device)
model.eval()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1536)
    (wpe): Embedding(2048, 1536)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1536, out_features=50257, bias=False)
)

In [109]:
prompt = "Клиент: Мужчина, 30 лет, женат, заработная плата 45 000 рублей, высшее образование."

prompt = prompt + '#'
encoded_prompt = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").to(device)

In [110]:
pad_token_id = tokenizer.encode('<pad>', add_special_tokens=False)[0]

In [111]:
output_sequences = model.generate(input_ids=encoded_prompt,
                                     max_length=200,
                                     repetition_penalty=1.15,
                                     do_sample=True,
                                     top_k=30,
                                     top_p=0.9,
                                     temperature=0.2,
                                     num_beams=1,
                                     no_repeat_ngram_size=15,
                                      pad_token_id=pad_token_id)


In [112]:
stop_token = '</s>'

In [113]:
output_sequences = output_sequences.tolist()

In [114]:
text = tokenizer.decode(output_sequences[0], clean_up_tokenization_spaces=True)
    
if stop_token in text:
    text = text[: text.find(stop_token)]

In [115]:
text = text[text.index('#')+1:].strip()

In [116]:
text = text.replace('\u0301', '').split('\u2010')

In [117]:
print(' '.join(text[::-1]))

Сформируй банковское предложение для клиента: Мужчина, 30 лет, женат, заработная плата 45 000 рублей, высшее образование
